# DRF Pagination

## Why Pagination?

Returning all records in a single response is impractical for large datasets — it is slow, memory-intensive, and transfers too much data to the client. Pagination divides results into pages.

DRF provides three built-in pagination styles:
- `PageNumberPagination` — `?page=2`
- `LimitOffsetPagination` — `?limit=10&offset=20`
- `CursorPagination` — opaque cursor token (best for real-time feeds)


## PageNumberPagination

The most common style. Clients request a page by number.

```python
# books/pagination.py
from rest_framework.pagination import PageNumberPagination

class StandardPagePagination(PageNumberPagination):
    page_size             = 10          # default items per page
    page_size_query_param = 'page_size' # client can override: ?page_size=25
    max_page_size         = 100
    page_query_param      = 'page'      # default — ?page=2
```

```python
# views.py
class BookViewSet(viewsets.ModelViewSet):
    queryset         = Book.objects.all()
    serializer_class = BookSerializer
    pagination_class = StandardPagePagination
```

Response format:
```json
{
  "count":    100,
  "next":     "http://api.example.com/books/?page=3",
  "previous": "http://api.example.com/books/?page=1",
  "results":  [...]
}
```


## LimitOffsetPagination

Clients control how many items to fetch (`limit`) and where to start (`offset`).

```python
from rest_framework.pagination import LimitOffsetPagination

class LargeResultsPagination(LimitOffsetPagination):
    default_limit = 20
    max_limit     = 200
```

```
GET /books/?limit=5&offset=10   # items 11–15
```

Response format:
```json
{
  "count":    100,
  "next":     "http://api.example.com/books/?limit=5&offset=15",
  "previous": "http://api.example.com/books/?limit=5&offset=5",
  "results":  [...]
}
```

`LimitOffsetPagination` is useful when the client needs to jump to an arbitrary position (e.g. infinite scroll loading a specific offset).


## CursorPagination

Uses an opaque cursor instead of a page number. Clients cannot skip pages — they can only move forward or backward. This guarantees stable pagination over changing data.

```python
from rest_framework.pagination import CursorPagination

class CreatedAtCursorPagination(CursorPagination):
    page_size = 20
    ordering  = '-created_at'   # must match a model field
```

```json
{
  "next":     "http://api.example.com/books/?cursor=cD0yMDIz...",
  "previous": null,
  "results":  [...]
}
```

Use `CursorPagination` for real-time feeds or timelines where new objects are frequently inserted.


## Global Default Pagination

Apply one style to all views without setting `pagination_class` on each:

```python
# settings.py
REST_FRAMEWORK = {
    'DEFAULT_PAGINATION_CLASS': 'rest_framework.pagination.PageNumberPagination',
    'PAGE_SIZE': 20,
}
```

To disable pagination for a specific view, set `pagination_class = None` on that view.


## Custom Pagination Response

Override `get_paginated_response()` to reshape the response envelope:

```python
from rest_framework.pagination import PageNumberPagination
from rest_framework.response import Response

class CustomPagination(PageNumberPagination):
    page_size = 10

    def get_paginated_response(self, data):
        return Response({
            'pagination': {
                'total_items': self.page.paginator.count,
                'total_pages': self.page.paginator.num_pages,
                'current_page': self.page.number,
                'next': self.get_next_link(),
                'previous': self.get_previous_link(),
            },
            'results': data,
        })
```


## Testing Pagination

```python
from rest_framework.test import APITestCase
from books.models import Author, Book

class PaginationTests(APITestCase):
    def setUp(self):
        author = Author.objects.create(name='Alice', slug='alice')
        for i in range(25):
            Book.objects.create(title=f'Book {i}', author=author, price=10)

    def test_default_page_size(self):
        resp = self.client.get('/api/books/')
        self.assertEqual(resp.status_code, 200)
        self.assertEqual(len(resp.data['results']), 10)
        self.assertIn('next', resp.data)
        self.assertEqual(resp.data['count'], 25)

    def test_page_2(self):
        resp = self.client.get('/api/books/?page=2')
        self.assertEqual(resp.status_code, 200)
        self.assertEqual(len(resp.data['results']), 10)

    def test_last_page(self):
        resp = self.client.get('/api/books/?page=3')
        self.assertEqual(len(resp.data['results']), 5)
        self.assertIsNone(resp.data['next'])
```


## Summary

- `PageNumberPagination` is the simplest; clients request `?page=N`.
- `LimitOffsetPagination` lets clients pick how many items and where to start: `?limit=10&offset=20`.
- `CursorPagination` uses opaque tokens for stable pagination over live data; clients cannot skip pages.
- Configure a global default in `DEFAULT_PAGINATION_CLASS` and `PAGE_SIZE` in `REST_FRAMEWORK` settings.
- Override `get_paginated_response()` to customize the response envelope shape.
- Set `pagination_class = None` on a view to disable pagination for that endpoint only.
